# 🧠 Word2Vec From Scratch — Workshop Demo

Based on: **Mikolov et al. (2013) — *Efficient Estimation of Word Representations in Vector Space***

---

### What we'll build today:
1. Load & preprocess the IMDB dataset
2. Build a vocabulary (~10k words)
3. Generate Skip-gram (target, context) training pairs
4. Train Skip-gram with **Negative Sampling** from scratch in PyTorch
5. Query similar words using cosine similarity

> 💡 **Goal:** Train in **under 5 minutes on CPU** using a subset of IMDB.

---
### Install dependencies (run once)

In [ ]:
# Run this cell first to install required packages
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "datasets", "torch", "numpy", "matplotlib",
                       "scikit-learn", "tqdm", "-q"])
print("✅ All packages installed!")

---
## 📦 Step 1 — Imports

In [ ]:
import re
import random
import numpy as np
from collections import Counter
from itertools import chain

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 📥 Step 2 — Load IMDB Dataset

We'll use the HuggingFace `datasets` library to load IMDB. To keep training fast, we take only the **first 5,000 reviews**.

In [ ]:
print("Loading IMDB dataset...")
dataset = load_dataset("imdb", split="train")

# --- Workshop knob: reduce this number for faster runs ---
NUM_REVIEWS = 5_000

raw_texts = [dataset[i]["text"] for i in range(NUM_REVIEWS)]

print(f"✅ Loaded {len(raw_texts):,} reviews")
print("\nSample review (first 200 chars):")
print(raw_texts[0][:200])

---
## 🔧 Step 3 — Preprocessing

Keep it simple:
- Lowercase everything
- Remove non-alphabetic characters
- Tokenize by whitespace

In [ ]:
def preprocess(text: str) -> list[str]:
    """Lowercase, strip HTML/special chars, whitespace-tokenize."""
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)          # remove HTML tags
    text = re.sub(r"[^a-z\s]", " ", text)          # keep only a-z and spaces
    text = re.sub(r"\s+", " ", text).strip()        # collapse whitespace
    return text.split()

tokenized = [preprocess(t) for t in raw_texts]

# Quick sanity check
print("Raw   :", raw_texts[0][:80])
print("Tokens:", tokenized[0][:15])
total_tokens = sum(len(s) for s in tokenized)
print(f"\nTotal tokens across {NUM_REVIEWS:,} reviews: {total_tokens:,}")

---
## 📚 Step 4 — Build Vocabulary (~10k words)

We keep only the **10,000 most frequent** words. Everything else becomes `<UNK>`.

> 📖 *The paper limits vocabulary to the most frequent words to keep the model tractable.*

In [ ]:
VOCAB_SIZE = 10_000   # Workshop knob
MIN_COUNT  = 5        # Ignore words that appear fewer than this many times

# Count every token
all_tokens = list(chain.from_iterable(tokenized))
freq = Counter(all_tokens)

# Keep top VOCAB_SIZE words with at least MIN_COUNT occurrences
vocab_words = [w for w, c in freq.most_common(VOCAB_SIZE) if c >= MIN_COUNT]
vocab_words = ["<UNK>"] + vocab_words   # index 0 → <UNK>

word2idx = {w: i for i, w in enumerate(vocab_words)}
idx2word = {i: w for w, i in word2idx.items()}

VOCAB_SIZE_ACTUAL = len(word2idx)
print(f"Vocabulary size: {VOCAB_SIZE_ACTUAL:,} (including <UNK>)")
print("Top 20 words:", vocab_words[1:21])

In [ ]:
# Replace out-of-vocab tokens with <UNK>
UNK_IDX = word2idx["<UNK>"]

encoded = [
    [word2idx.get(w, UNK_IDX) for w in sentence]
    for sentence in tokenized
]

# Quick coverage check
unk_count = sum(idx == UNK_IDX for sent in encoded for idx in sent)
unk_pct   = 100 * unk_count / total_tokens
print(f"<UNK> tokens: {unk_count:,} ({unk_pct:.1f}% of all tokens)")

---
## 🪟 Step 5 — Generate Skip-gram (target, context) Pairs

The **Skip-gram** model takes a **center word** and tries to predict the surrounding **context words** within a window.

```
Sentence : "the movie was really great"
Window=2  : center='was' → context=['the','movie','really','great']
Pairs     : (was, the), (was, movie), (was, really), (was, great)
```

> 📖 *Mikolov et al. use C=10 for max window. We use 2–3 for speed.*

In [ ]:
WINDOW_SIZE = 2   # Workshop knob: try 2 or 3

def generate_skipgram_pairs(encoded_sentences: list[list[int]],
                             window: int = 2,
                             unk_idx: int = 0) -> list[tuple[int, int]]:
    """
    For each center word, emit (center_idx, context_idx) pairs.
    Skips <UNK> tokens as centers (but allows them as context).
    """
    pairs = []
    for sentence in encoded_sentences:
        n = len(sentence)
        for center_pos, center_word in enumerate(sentence):
            if center_word == unk_idx:
                continue  # skip <UNK> centers
            # Context window: [center-window, center+window] excluding center itself
            for offset in range(-window, window + 1):
                if offset == 0:
                    continue
                ctx_pos = center_pos + offset
                if 0 <= ctx_pos < n:
                    pairs.append((center_word, sentence[ctx_pos]))
    return pairs

print("Generating Skip-gram pairs...")
all_pairs = generate_skipgram_pairs(encoded, window=WINDOW_SIZE, unk_idx=UNK_IDX)
print(f"✅ Generated {len(all_pairs):,} (target, context) pairs")
print("\nFirst 5 pairs (word indices):")
for t, c in all_pairs[:5]:
    print(f"  target='{idx2word[t]}'  context='{idx2word[c]}'")

---
## ⚙️ Step 6 — Negative Sampling Setup

Instead of computing a full softmax over the entire vocabulary (slow!), **Negative Sampling** trains the model to:
- Predict **1** for real (target, context) pairs
- Predict **0** for `K` randomly sampled *noise* (target, random_word) pairs

Words are sampled proportional to **frequency^(3/4)** — a trick from the paper that slightly
upsamples rare words.

> 📖 *Mikolov et al. recommend K=5–20 for small datasets; we use K=5.*

In [ ]:
NEG_SAMPLES = 5   # K negative samples per positive pair

# Build the noise distribution (freq^0.75 trick from the paper)
# We only sample from in-vocab words (excluding <UNK> at index 0)
counts = np.array([freq.get(idx2word.get(i, ""), 1) for i in range(VOCAB_SIZE_ACTUAL)],
                  dtype=np.float32)
counts[UNK_IDX] = 0   # never sample <UNK> as a negative
noise_dist = counts ** 0.75
noise_dist /= noise_dist.sum()

print("Noise distribution built.")
print(f"Probability of top-3 words as negatives:")
top3 = np.argsort(noise_dist)[::-1][:3]
for idx in top3:
    print(f"  '{idx2word[idx]}': {noise_dist[idx]:.5f}")

---
## 🗃️ Step 7 — PyTorch Dataset

In [ ]:
class SkipGramDataset(Dataset):
    """
    Each item is (target_idx, context_idx, neg_idx_1, ..., neg_idx_K).
    Negatives are pre-sampled per item for speed.
    """
    def __init__(self, pairs: list[tuple[int, int]],
                 noise_dist: np.ndarray,
                 k: int = 5):
        self.pairs      = pairs
        self.noise_dist = noise_dist
        self.k          = k
        self.vocab_size = len(noise_dist)
        # Pre-sample ALL negatives up front (much faster than per __getitem__)
        print("Pre-sampling negatives...")
        self.negatives = np.random.choice(
            self.vocab_size,
            size=(len(pairs), k),
            p=noise_dist
        )
        print("✅ Negatives ready.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        target, context = self.pairs[idx]
        negs = self.negatives[idx]          # shape (K,)
        return (
            torch.tensor(target,  dtype=torch.long),
            torch.tensor(context, dtype=torch.long),
            torch.tensor(negs,    dtype=torch.long),
        )

dataset_sg = SkipGramDataset(all_pairs, noise_dist, k=NEG_SAMPLES)
print(f"Dataset size: {len(dataset_sg):,} samples")

---
## 🏗️ Step 8 — Skip-gram Model

The model has **two embedding tables**:
- `target_embeddings` — the "input" / center-word vectors (what we'll use after training)
- `context_embeddings` — the "output" / context-word vectors

The loss is the **Negative Sampling objective** (binary cross-entropy):

$$\mathcal{L} = -\log\sigma(v_c \cdot v_t) - \sum_{i=1}^{K} \log\sigma(-v_{n_i} \cdot v_t)$$

> 📖 *The paper uses a log-linear model — no hidden layers. That's exactly what we implement here.*

In [ ]:
class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        # Center-word ("input") embeddings
        self.target_emb  = nn.Embedding(vocab_size, embed_dim)
        # Context-word ("output") embeddings
        self.context_emb = nn.Embedding(vocab_size, embed_dim)

        # Init with small uniform values (common practice)
        nn.init.uniform_(self.target_emb.weight,  -0.5 / embed_dim, 0.5 / embed_dim)
        nn.init.uniform_(self.context_emb.weight, -0.5 / embed_dim, 0.5 / embed_dim)

    def forward(self, targets, contexts, negatives):
        """
        targets   : (B,)
        contexts  : (B,)
        negatives : (B, K)
        Returns scalar loss.
        """
        t_vecs = self.target_emb(targets)    # (B, D)
        c_vecs = self.context_emb(contexts)  # (B, D)
        n_vecs = self.context_emb(negatives) # (B, K, D)

        # Positive score: dot product of target & context
        pos_score = torch.sum(t_vecs * c_vecs, dim=1)  # (B,)
        pos_loss  = torch.nn.functional.logsigmoid(pos_score)

        # Negative scores: dot product of target & each negative
        # t_vecs unsqueezed → (B, 1, D), bmm → (B, K, 1) → squeeze → (B, K)
        neg_scores = torch.bmm(n_vecs, t_vecs.unsqueeze(2)).squeeze(2)  # (B, K)
        neg_loss   = torch.nn.functional.logsigmoid(-neg_scores).sum(dim=1)  # (B,)

        # We minimise the negative of the objective
        loss = -(pos_loss + neg_loss).mean()
        return loss

    def get_word_vector(self, idx: int) -> np.ndarray:
        """Return the target embedding for a single word index."""
        with torch.no_grad():
            return self.target_emb.weight[idx].cpu().numpy()

---
## 🚀 Step 9 — Training Loop

**Hyperparameter knobs** (adjust these during the workshop!):

| Parameter | Value | Notes |
|-----------|-------|-------|
| `EMBED_DIM` | 100 | Vector dimensionality |
| `EPOCHS` | 3 | 1 epoch = one pass over all pairs |
| `BATCH_SIZE` | 512 | Larger = faster but needs more RAM |
| `LR` | 0.025 | Linear decay to 0 (as in the paper) |

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────
EMBED_DIM  = 100    # Word vector dimensionality
EPOCHS     = 3      # Number of training epochs
BATCH_SIZE = 512    # Samples per gradient update
LR         = 0.025  # Starting learning rate (linearly decayed)
# ─────────────────────────────────────────────────────────────

loader = DataLoader(dataset_sg, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

model     = SkipGramNegSampling(VOCAB_SIZE_ACTUAL, EMBED_DIM).to(DEVICE)
optimizer = optim.SGD(model.parameters(), lr=LR)

total_steps   = EPOCHS * len(loader)
step          = 0
loss_history  = []

print(f"Training for {EPOCHS} epoch(s), {len(loader):,} batches each")
print(f"Total gradient steps: {total_steps:,}")
print("-" * 55)

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True)

    for targets, contexts, negatives in pbar:
        # Linear LR decay — exactly as described in the paper
        progress = step / total_steps
        current_lr = LR * max(0.0001, 1.0 - progress)
        for pg in optimizer.param_groups:
            pg["lr"] = current_lr

        targets   = targets.to(DEVICE)
        contexts  = contexts.to(DEVICE)
        negatives = negatives.to(DEVICE)

        optimizer.zero_grad()
        loss = model(targets, contexts, negatives)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        step       += 1
        loss_history.append(loss.item())

        if step % 500 == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{current_lr:.5f}")

    avg = epoch_loss / len(loader)
    print(f"  ✅ Epoch {epoch} done — avg loss: {avg:.4f}")

print("\n🎉 Training complete!")

In [ ]:
# Plot training loss (smoothed)
def smooth(values, window=200):
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(smooth(loss_history), color="steelblue", linewidth=1.5)
plt.title("Skip-gram Negative Sampling — Training Loss (smoothed)", fontsize=13)
plt.xlabel("Gradient step")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🔍 Step 10 — Query: `most_similar(word, k=5)`

We compare word vectors using **cosine similarity**:

$$\text{sim}(u, v) = \frac{u \cdot v}{\|u\| \|v\|}$$

Higher = more similar.

In [ ]:
# Extract the full embedding matrix (V × D) once
model.eval()
with torch.no_grad():
    embeddings = model.target_emb.weight.cpu().numpy()   # shape (V, D)

# Pre-normalise every vector for fast cosine similarity via dot product
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1, norms)   # avoid div-by-zero
norm_embeddings = embeddings / norms

def most_similar(word: str, k: int = 5) -> list[tuple[str, float]]:
    """
    Return the k most similar words to `word` by cosine similarity.

    Parameters
    ----------
    word : str  — must be in the vocabulary
    k    : int  — number of neighbours to return

    Returns
    -------
    List of (word, similarity_score) tuples, sorted descending.
    """
    word = word.lower()
    if word not in word2idx:
        print(f"⚠️  '{word}' is not in the vocabulary.")
        return []

    idx = word2idx[word]
    query_vec = norm_embeddings[idx]                       # (D,)
    scores    = norm_embeddings @ query_vec                # (V,) cosine sims
    scores[idx] = -np.inf                                  # exclude the word itself

    top_k_idx = np.argpartition(scores, -k)[-k:]          # fast top-k
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]

    return [(idx2word[i], float(scores[i])) for i in top_k_idx]

print("most_similar() is ready! 🎉")

In [ ]:
# ── Try it out! ──────────────────────────────────────────────
test_words = ["good", "bad", "film", "love", "scary"]

for word in test_words:
    results = most_similar(word, k=5)
    if results:
        print(f"\n🔎 most_similar('{word}'):")
        for neighbour, score in results:
            print(f"   {neighbour:<15}  {score:.4f}")

---
## ➕ Bonus — Vector Arithmetic

The classic demo from the paper:

> `vector("king") - vector("man") + vector("woman")` ≈ `vector("queen")`

Our model was only trained on 5k reviews for a few minutes, so don't expect perfect analogies — but try it!

In [ ]:
def analogy(pos1: str, neg1: str, pos2: str, k: int = 5) -> list[tuple[str, float]]:
    """
    Answers: pos1 - neg1 + pos2 = ?
    Example: analogy('king', 'man', 'woman')  →  'queen' (hopefully!)
    """
    words = [pos1.lower(), neg1.lower(), pos2.lower()]
    for w in words:
        if w not in word2idx:
            print(f"⚠️  '{w}' not in vocabulary.")
            return []

    v = (norm_embeddings[word2idx[pos1]] -
         norm_embeddings[word2idx[neg1]] +
         norm_embeddings[word2idx[pos2]])
    v = v / np.linalg.norm(v)

    scores = norm_embeddings @ v
    # Exclude the three input words
    for w in words:
        scores[word2idx[w]] = -np.inf

    top_k = np.argpartition(scores, -k)[-k:]
    top_k = top_k[np.argsort(scores[top_k])[::-1]]
    return [(idx2word[i], float(scores[i])) for i in top_k]


print("Analogy: good - great + terrible?")
for w, s in analogy("good", "great", "terrible"):
    print(f"  {w:<15}  {s:.4f}")

print("\nAnalogy: love - loved + hated?")
for w, s in analogy("love", "loved", "hated"):
    print(f"  {w:<15}  {s:.4f}")

---
## 🗺️ Bonus — 2D Visualisation with PCA

Project the 100-D word vectors down to 2D and plot clusters.

In [ ]:
from sklearn.decomposition import PCA

# Words to visualise — feel free to swap these!
vis_words = [
    # Sentiment
    "great", "excellent", "good", "wonderful", "amazing",
    "bad",   "terrible",  "poor", "awful",     "boring",
    # Film terms
    "film",  "movie",     "plot", "story",     "character",
    "acting","director",  "scene","cast",      "script",
]
vis_words = [w for w in vis_words if w in word2idx]  # keep in-vocab only

vecs = np.array([norm_embeddings[word2idx[w]] for w in vis_words])

pca    = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(vecs)

plt.figure(figsize=(11, 7))
colors = (["#2196F3"] * 10 + ["#FF5722"] * 10)[:len(vis_words)]
plt.scatter(coords[:, 0], coords[:, 1], c=colors, s=80, zorder=3)

for i, word in enumerate(vis_words):
    plt.annotate(word, (coords[i, 0] + 0.01, coords[i, 1] + 0.01), fontsize=10)

plt.title("Word Embeddings — PCA Projection (blue=sentiment, orange=film terms)", fontsize=12)
plt.axhline(0, color="grey", lw=0.5)
plt.axvline(0, color="grey", lw=0.5)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

---
## 💾 Save & Load Embeddings

In [ ]:
# Save model weights
torch.save({
    "model_state": model.state_dict(),
    "word2idx":    word2idx,
    "idx2word":    idx2word,
    "embed_dim":   EMBED_DIM,
}, "word2vec_imdb.pt")
print("✅ Saved to word2vec_imdb.pt")

# Load it back
checkpoint = torch.load("word2vec_imdb.pt", map_location="cpu")
loaded_model = SkipGramNegSampling(len(checkpoint["word2idx"]), checkpoint["embed_dim"])
loaded_model.load_state_dict(checkpoint["model_state"])
loaded_model.eval()
print("✅ Loaded model successfully")

---
## 📝 Workshop Recap

| Step | What we did |
|------|-------------|
| 1 | Loaded IMDB reviews from HuggingFace |
| 2 | Preprocessed: lowercase, strip HTML, whitespace-tokenise |
| 3 | Built vocabulary of the 10k most frequent words |
| 4 | Generated (target, context) Skip-gram pairs with window=2 |
| 5 | Implemented Negative Sampling with K=5 and freq^0.75 noise |
| 6 | Trained a 2-embedding-table model (no hidden layers!) in PyTorch |
| 7 | Queried nearest neighbours using cosine similarity |
| 8 | Visualised clusters with PCA |

### 🔬 Things to try next:
- Increase `NUM_REVIEWS` to 25k (full dataset) and retrain
- Increase `EMBED_DIM` to 300
- Increase `WINDOW_SIZE` to 5
- Switch to **CBOW**: predict center word from context sum
- Use subword tokenisation (BPE) instead of whitespace

---
*Based on: Mikolov et al. (2013), "Efficient Estimation of Word Representations in Vector Space"*